# Tugas Akhir PCVK - Verifikasi Dataset SIPaKMeD

Notebook ini memverifikasi kelayakan dataset SIPaKMeD dan mengisi **Tabel 8** pada lembar studi literatur. Dirancang untuk **Google Colab**, tetapi tetap bisa dijalankan lokal.

### Perbaikan dari versi sebelumnya

| Masalah versi lama | Perbaikan |
|---|---|
| `rglob('*.bmp')` menghitung citra klaster **dan** sel tunggal, hasilnya 5015 | Hanya `CROPPED/` yang dihitung, hasilnya 4049 (angka resmi) |
| `dataset_root` dipakai sebelum didefinisikan sehingga `NameError` | Konfigurasi dan deteksi path dikumpulkan di satu sel paling awal |
| Ekstrak seluruh `.7z` menghasilkan 9,7 GB, Colab kehabisan disk | Unduh hanya `CROPPED/` dari mirror, sekitar 590 MB |
| Statistik resolusi mencampur dua populasi citra | Statistik dipisah: sel tunggal dan klaster |
| `format_counter` bisa kosong sehingga `IndexError` | Format dihitung dari file yang benar-benar dibaca |
| Grid contoh bergeser bila ada folder kelas kosong | Indeks mengikuti kelas yang benar-benar punya citra |
| Tidak ada bukti ukuran file, integritas, dan ID klaster | Ditambahkan: ukuran disk, cek file rusak, cek hash, analisis ID klaster |
| Tidak bisa dijalankan di Colab kosong | Unduhan otomatis dari mirror, tanpa Google Drive |

**Luaran:** `distribusi_sipakmed.png`, `sample_citra_sipakmed.png`, `verifikasi_dataset.txt`, `verifikasi_dataset.csv`, `ringkasan_dataset.json`

## 1. Setup

In [ ]:
import os
import re
import sys
import json
import time
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]} | Colab: {IN_COLAB}")
print(f"numpy {np.__version__} | matplotlib {plt.matplotlib.__version__} | Pillow {Image.__version__}")

## 2. Konfigurasi

Angka resmi dari halaman SIPaKMeD dan Plissiti et al. (2018) dipakai sebagai **pembanding**, bukan sebagai sumber data.

Tentang sumber unduhan: situs resmi University of Ioannina masih menyediakan kelima arsip, tetapi kecepatannya diukur hanya sekitar 36 KB/s sehingga 4,8 GB tidak realistis. Notebook memakai mirror HuggingFace `rjaditya/sipakmed-dataset` yang isinya sudah diverifikasi identik byte-per-byte dengan rilis resmi (lihat bagian 11). Sitasi di laporan tetap mengacu ke Plissiti et al. (2018).

In [ ]:
REPO_HF = "rjaditya/sipakmed-dataset"

KELAS_RESMI = {
    "im_Dyskeratotic": 813,
    "im_Koilocytotic": 825,
    "im_Metaplastic": 793,
    "im_Parabasal": 787,
    "im_Superficial-Intermediate": 831,
}
TOTAL_RESMI = 4049
KLASTER_RESMI = 966

# Folder tujuan. Di Colab /content hilang saat runtime mati, jadi unduhan
# diulang tiap sesi baru. Kalau ingin menyimpan antar sesi, arahkan ke
# folder Google Drive yang sudah di-mount.
if IN_COLAB:
    TUJUAN = Path("/content/sipakmed")
else:
    TUJUAN = Path("./sipakmed")

# Kandidat lokasi dataset yang sudah ada (dipakai lebih dulu bila ditemukan)
KANDIDAT_ROOT = [
    os.environ.get("SIPAKMED_ROOT", ""),
    str(TUJUAN),
    "/content/drive/MyDrive/PCVK/sipakmed",
    "/content/dataset/sipakmed",
    "./sipakmed",
    "../sipakmed",
]

print(f"Folder tujuan unduhan : {TUJUAN}")

## 3. Unduh Dataset (hanya `CROPPED/`)

Tiap arsip asli berisi dua populasi citra yang berbeda:

```
im_Parabasal/
├── 001.bmp              citra klaster, 2048x1536, ~9 MB per file
├── 001_cyt01.dat        koordinat poligon sitoplasma
├── 001_nuc01.dat        koordinat poligon nukleus
└── CROPPED/001_01.bmp   sel tunggal, ~150 KB per file   <- dipakai proyek
```

Mengunduh semuanya butuh 9,7 GB. Proyek hanya memakai sel tunggal, jadi hanya `CROPPED/*.bmp` yang diambil (sekitar 590 MB, 4049 file).

Catatan teknis: unduhan dilakukan paralel dengan `curl`. Batas paralel 24 sempat memicu rate limit (HTTP 429) di HuggingFace, sehingga dipakai 16 dengan pemeriksaan ulang. File yang gagal dikenali dari ukurannya yang tidak wajar (di bawah 1 KB) lalu diunduh ulang.

In [ ]:
def sudah_lengkap(root):
    """True bila kelima kelas sudah punya jumlah file yang benar."""
    if not root.is_dir():
        return False
    for nama, harus in KELAS_RESMI.items():
        d = root / nama / "CROPPED"
        if not d.is_dir():
            return False
        berkas = list(d.glob("*.bmp"))
        if len(berkas) != harus:
            return False
        if any(f.stat().st_size < 1000 for f in berkas):
            return False
    return True


def daftar_berkas_hf(repo=REPO_HF):
    from huggingface_hub import list_repo_files
    semua = list_repo_files(repo, repo_type="dataset")
    hasil = {}
    for nama in KELAS_RESMI:
        pola = f"images/{nama}/{nama}/CROPPED/"
        hasil[nama] = [
            f for f in semua if f.startswith(pola) and f.lower().endswith(".bmp")
        ]
    return hasil


def unduh_paralel(daftar, tujuan, paralel=8, bat=300):
    """Unduh daftar path HF ke tujuan/<kelas>/CROPPED/ memakai curl paralel."""
    dasar = f"https://huggingface.co/datasets/{REPO_HF}/resolve/main/"
    for nama, berkas in daftar.items():
        keluar = tujuan / nama / "CROPPED"
        keluar.mkdir(parents=True, exist_ok=True)
        for i in range(0, len(berkas), bat):
            potong = berkas[i:i + bat]
            subprocess.run(
                ["curl", "-sL", "--parallel", "--parallel-max", str(paralel),
                 "--remote-name-all", "--output-dir", str(keluar)]
                + [dasar + f for f in potong],
                check=True,
            )


def bisa_dibaca(f, minimal=1000):
    """False bila file terlalu kecil atau bukan citra yang sah."""
    try:
        if f.stat().st_size < minimal:
            return False
        with Image.open(f) as im:
            im.verify()
        return True
    except Exception:
        return False


def bersihkan_dan_ulang(tujuan, minimal=1000, paralel=4, percobaan=5):
    """Hapus file yang terlalu kecil (halaman error) lalu unduh ulang."""
    for putaran in range(1, percobaan + 1):
        rusak = {}
        for nama in KELAS_RESMI:
            keluar = tujuan / nama / "CROPPED"
            if not keluar.is_dir():
                continue
            kecil = [f for f in keluar.glob("*.bmp") if not bisa_dibaca(f, minimal)]
            if kecil:
                rusak[nama] = kecil
        if not rusak:
            return 0
        total = sum(len(v) for v in rusak.values())
        print(f"  Putaran {putaran}: {total} file gagal, mengunduh ulang...")
        for nama, kecil in rusak.items():
            keluar = tujuan / nama / "CROPPED"
            for f in kecil:
                f.unlink()
            dasar = f"https://huggingface.co/datasets/{REPO_HF}/resolve/main/"
            subprocess.run(
                ["curl", "-sL", "--parallel", "--parallel-max", str(paralel),
                 "--remote-name-all", "--output-dir", str(keluar)]
                + [f"{dasar}images/{nama}/{nama}/CROPPED/{f.name}" for f in kecil],
                check=True,
            )
        time.sleep(5)
    return sum(len([f for f in (tujuan / n / "CROPPED").glob("*.bmp") if bisa_dibaca(f)])
               for n in KELAS_RESMI if (tujuan / n / "CROPPED").is_dir())


if sudah_lengkap(TUJUAN):
    print(f"Dataset sudah lengkap di {TUJUAN}, unduhan dilewati.")
else:
    print(f"Mengunduh 4049 citra sel tunggal ke {TUJUAN} ...")
    mulai = time.time()
    daftar = daftar_berkas_hf()
    for nama, berkas in daftar.items():
        print(f"  {nama:30s} {len(berkas):4d} file")
    unduh_paralel(daftar, TUJUAN)
    sisa = bersihkan_dan_ulang(TUJUAN)
    print(f"Selesai dalam {time.time() - mulai:.0f} detik")

## 4. Ekstraksi Arsip `.7z` (hanya bila ada arsip lokal)

Dilewati bila dataset sudah berbentuk folder. Berguna kalau kelompok memakai arsip resmi dari University of Ioannina yang sudah diunduh sebelumnya.

In [ ]:
arsip = sorted(TUJUAN.glob("*.7z")) if TUJUAN.is_dir() else []
butuh = [a for a in arsip if not (TUJUAN / a.stem / "CROPPED").is_dir()]

if not arsip:
    print("Tidak ada arsip .7z. Dilewati.")
elif not butuh:
    print(f"Ada {len(arsip)} arsip, semuanya sudah punya CROPPED/. Dilewati.")
else:
    try:
        import py7zr
    except ImportError:
        print("py7zr belum terpasang. Di Colab jalankan: !pip install -q py7zr")
        raise
    for a in butuh:
        with py7zr.SevenZipFile(a, "r") as z:
            target = [n for n in z.getnames()
                      if n.startswith(f"{a.stem}/CROPPED/") and n.lower().endswith(".bmp")]
            z.extract(path=TUJUAN, targets=target)
        print(f"  {a.stem}: {len(target)} citra")

## 5. Deteksi Lokasi Dataset

In [ ]:
def cari_root(kandidat):
    for k in kandidat:
        if not k:
            continue
        p = Path(k).expanduser()
        if p.is_dir() and any((p / n).is_dir() for n in KELAS_RESMI):
            return p.resolve()
    return None


ROOT = cari_root(KANDIDAT_ROOT)

if ROOT is None:
    print("Dataset tidak ditemukan. Pilihan:")
    print("  Colab : jalankan ulang bagian 3 (unduh otomatis)")
    print("  Lokal : ekstrak arsip ke <repo>/sipakmed")
    print("  Manual: set SIPAKMED_ROOT=/path/ke/sipakmed")
assert ROOT is not None, "Lokasi dataset tidak ditemukan."
print(f"ROOT dataset: {ROOT}")

## 6. Inventaris Struktur Folder

Menghitung ketiga jenis file secara terpisah supaya tidak tertukar seperti pada versi lama. Citra klaster dan file `.dat` opsional: kalau hanya `CROPPED/` yang diunduh, kolomnya akan bernilai 0 dan itu wajar.

In [ ]:
def inventaris(root):
    baris = []
    for nama in KELAS_RESMI:
        d = root / nama
        if not d.is_dir():
            continue
        crop = sorted(d.glob("CROPPED/*.bmp"))
        klaster = sorted(d.glob("*.bmp"))
        baris.append({
            "kelas": nama,
            "sel_tunggal": len(crop),
            "klaster": len(klaster),
            "dat": len(list(d.glob("*.dat"))),
            "mb_sel_tunggal": sum(f.stat().st_size for f in crop) / 1e6,
            "mb_klaster": sum(f.stat().st_size for f in klaster) / 1e6,
        })
    return baris


INV = inventaris(ROOT)
print(f"{'Kelas':30s} {'Sel tunggal':>12s} {'Klaster':>8s} {'.dat':>6s} {'MB crop':>9s}")
print("-" * 70)
for r in INV:
    print(f"{r['kelas']:30s} {r['sel_tunggal']:12d} {r['klaster']:8d} {r['dat']:6d} "
          f"{r['mb_sel_tunggal']:9.1f}")
print("-" * 70)

tot_crop = sum(r["sel_tunggal"] for r in INV)
tot_klaster = sum(r["klaster"] for r in INV)
tot_dat = sum(r["dat"] for r in INV)
print(f"{'TOTAL':30s} {tot_crop:12d} {tot_klaster:8d} {tot_dat:6d} "
      f"{sum(r['mb_sel_tunggal'] for r in INV):9.1f}")
print()
print("Sel tunggal (CROPPED/) adalah data yang dipakai proyek.")
print("Citra klaster hanya konteks sumber, tidak dipakai.")

## 7. Verifikasi terhadap Angka Resmi

Ini bagian yang menjawab keraguan "5015 atau 4049". Notebook lama menjumlahkan dua populasi berbeda; sekarang tiap populasi dibandingkan dengan angkanya sendiri.

Jumlah 4049 **wajib** cocok. Jumlah klaster 966 hanya diperiksa bila citra klaster ikut diekstrak.

In [ ]:
print("Per kelas:")
semua_cocok = True
for r in INV:
    harap = KELAS_RESMI[r["kelas"]]
    ok = r["sel_tunggal"] == harap
    semua_cocok &= ok
    print(f"  [{'OK' if ok else 'BEDA'}] {r['kelas']:30s} resmi={harap:4d}  dihitung={r['sel_tunggal']:4d}")

cek_total = tot_crop == TOTAL_RESMI
cek_kelas = len(INV) == 5

print()
print(f"Total sel tunggal : {tot_crop:5d}  (resmi {TOTAL_RESMI})  -> {'COCOK' if cek_total else 'TIDAK COCOK'}")
print(f"Jumlah kelas      : {len(INV):5d}  (resmi 5)     -> {'COCOK' if cek_kelas else 'TIDAK COCOK'}")

if tot_klaster == 0:
    print("Total klaster     : tidak diperiksa (hanya CROPPED/ yang tersedia)")
    cek_klaster = True
else:
    cek_klaster = tot_klaster == KLASTER_RESMI
    print(f"Total klaster     : {tot_klaster:5d}  (resmi {KLASTER_RESMI})  -> "
          f"{'COCOK' if cek_klaster else 'TIDAK COCOK'}")

print()
print(f"Angka 5015 pada versi lama = {TOTAL_RESMI} sel tunggal + {KLASTER_RESMI} citra klaster.")

assert semua_cocok and cek_total and cek_kelas and cek_klaster, (
    "Jumlah tidak sesuai angka resmi. Periksa apakah folder CROPPED/ lengkap."
)
print()
print("Semua jumlah sesuai angka resmi.")

## 8. Distribusi Kelas dan Keseimbangan

Keseimbangan kelas menentukan apakah perlu penanganan imbalance (SMOTE atau class weight) pada tahap pelatihan.

In [ ]:
label = [r["kelas"].replace("im_", "").replace("-", " ").replace("_", " ") for r in INV]
jumlah = [r["sel_tunggal"] for r in INV]

fig, ax = plt.subplots(figsize=(11, 5))
warna = ["#2563EB", "#06B6D4", "#10B981", "#F59E0B", "#EF4444"]
bar = ax.bar(label, jumlah, color=warna[: len(label)], edgecolor="black", linewidth=1.2)
for b, n in zip(bar, jumlah):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 8, f"{n}",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_xlabel("Kelas sel", fontweight="bold")
ax.set_ylabel("Jumlah citra", fontweight="bold")
ax.set_title(f"Distribusi SIPaKMeD: {sum(jumlah)} sel tunggal pada {len(jumlah)} kelas",
             fontweight="bold")
ax.set_ylim(0, max(jumlah) * 1.15)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
plt.savefig("distribusi_sipakmed.png", dpi=150, bbox_inches="tight")
plt.show()

selisih = max(jumlah) - min(jumlah)
rasio = max(jumlah) / min(jumlah)
status = "SEIMBANG" if rasio < 1.2 else ("CUKUP SEIMBANG" if rasio < 1.5 else "TIDAK SEIMBANG")
print(f"Selisih max-min : {selisih} citra")
print(f"Rasio max/min   : {rasio:.3f} : 1")
print(f"Status          : {status}")
print()
print("Rasio di bawah 1,2 berarti penanganan imbalance khusus tidak diperlukan.")

## 9. ID Klaster dan Risiko Data Leakage

Nama file di `CROPPED/` berpola `NNN_MM.bmp`: `NNN` adalah nomor citra klaster asal, `MM` adalah nomor sel di dalam klaster tersebut. Contoh `001_07.bmp` berarti sel ke-7 dari klaster `001`.

Ini penting untuk dua hal:

- **Bukti struktur**: jumlah ID klaster unik seharusnya sama dengan jumlah citra klaster, yang berarti setiap sel bisa dilacak ke asalnya.
- **Pencegahan leakage**: sel dari klaster yang sama mirip secara visual. Kalau klaster yang sama muncul di data latih dan data uji, akurasi terlihat lebih tinggi dari kenyataan. Karena itu pembagian data memakai `GroupKFold` dengan grup berupa ID klaster.

Dataset ini **tidak** menyediakan ID pasien, sehingga pembagian per pasien tidak mungkin dilakukan dan harus dinyatakan sebagai keterbatasan di laporan.

In [ ]:
pola = re.compile(r"^(\d+)_(\d+)$")
klaster_per_kelas = {}

for r in INV:
    nama = r["kelas"]
    crop = sorted((ROOT / nama / "CROPPED").glob("*.bmp"))
    cocok = [pola.match(f.stem) for f in crop]
    id_klaster = [m.group(1) for m in cocok if m]
    tanpa_pola = len(crop) - len(id_klaster)
    c = Counter(id_klaster)
    klaster_per_kelas[nama] = c
    print(f"{nama:30s} sel={len(crop):4d} klaster={len(c):4d} "
          f"sel/klaster={min(c.values())}-{max(c.values())} tak_sesuai_pola={tanpa_pola}")

total_klaster_unik = sum(len(c) for c in klaster_per_kelas.values())
print()
print(f"Total klaster unik: {total_klaster_unik}")
if tot_klaster > 0:
    print(f"Citra klaster di disk: {tot_klaster} -> "
          f"{'COCOK' if total_klaster_unik == tot_klaster else 'TIDAK COCOK'}")
    yatim = 0
    for r in INV:
        nama = r["kelas"]
        ada = {p.stem for p in (ROOT / nama).glob("*.bmp")}
        yatim += len(set(klaster_per_kelas[nama]) - ada)
    print(f"ID klaster tanpa citra asal: {yatim}")
else:
    print("Citra klaster tidak tersedia, pemeriksaan silang dilewati.")

print()
print("Rencana pembagian data: GroupKFold(n_splits=5, groups=ID_klaster)")
print("Alternatif: GroupShuffleSplit 80:20 dengan grup yang sama.")

## 10. Resolusi dan Format Citra

Statistik dipisah per populasi. Versi lama mencampur keduanya sehingga muncul rentang aneh "80x70 sampai 2048x1536".

In [ ]:
def statistik_resolusi(files):
    lebar, tinggi = [], []
    for f in files:
        try:
            with Image.open(f) as im:
                lebar.append(im.size[0])
                tinggi.append(im.size[1])
        except Exception:
            pass
    return np.array(lebar), np.array(tinggi)


semua_crop = [f for r in INV for f in sorted((ROOT / r["kelas"] / "CROPPED").glob("*.bmp"))]
W, H = statistik_resolusi(semua_crop)

print(f"Sel tunggal (n={len(W)})")
print(f"  Lebar  : min {W.min():4d} | median {int(np.median(W)):4d} | rata-rata {W.mean():6.1f} | max {W.max():4d} px")
print(f"  Tinggi : min {H.min():4d} | median {int(np.median(H)):4d} | rata-rata {H.mean():6.1f} | max {H.max():4d} px")
print(f"  Rasio aspek median: {np.median(W / H):.2f}")
di_bawah = ((W < 224) | (H < 224)).mean() * 100
print(f"  Di bawah 224 px (ukuran input CNN pada A3 dan A4): {di_bawah:.1f}%")

format_counter = Counter()
for f in semua_crop:
    try:
        with Image.open(f) as im:
            format_counter[im.format] += 1
    except Exception:
        pass
print()
print(f"Format sel tunggal: {dict(format_counter)}")

if tot_klaster > 0:
    semua_klaster = [f for r in INV for f in sorted((ROOT / r["kelas"]).glob("*.bmp"))]
    KW, KH = statistik_resolusi(semua_klaster)
    print()
    print(f"Citra klaster (n={len(KW)}), tidak dipakai proyek")
    print(f"  Lebar  : {KW.min()}-{KW.max()} px | Tinggi: {KH.min()}-{KH.max()} px")
else:
    KW = KH = np.array([0])
    print()
    print("Citra klaster tidak tersedia (hanya CROPPED/ yang diunduh).")

print()
print("Selisih resolusi antar kelas membuat resize ke ukuran seragam wajib")
print("dilakukan pada tahap praproses sebelum ekstraksi fitur.")

## 11. Integritas File dan Kecocokan dengan Rilis Resmi

Dua pemeriksaan:

1. **Semua file terbaca.** Satu file rusak bisa menghentikan proses ekstraksi fitur di tengah jalan.
2. **Hash cocok dengan rilis resmi.** Sepuluh file (dua per kelas) dibandingkan SHA-256-nya dengan arsip resmi University of Ioannina. Ini yang menjawab pertanyaan "kok ambil dari mirror?".

Kalau hash berbeda, notebook memberi peringatan, bukan berhenti, karena kegagalan unduh biasanya sudah tertangkap pada bagian 3.

In [ ]:
HASH_RUJUKAN = {
    "im_Dyskeratotic/CROPPED/001_01.bmp": "5843e96a9d12d0f3ffe23a910ec888ac3c058a2a25d7ef809dd8fbe883a80906",
    "im_Dyskeratotic/CROPPED/001_02.bmp": "9ce55ee723cbba50d9f1160a057ae93f849c2f660d9b52c1ec09bced394e9155",
    "im_Koilocytotic/CROPPED/001_01.bmp": "39dfd49760694fb50e586be01a3c06d93b00aff6710623e773283f6bd8fd3855",
    "im_Koilocytotic/CROPPED/002_01.bmp": "317be092507558726c404ff77ed4aaac6dd02fd864254eaf051f43dc982cc817",
    "im_Metaplastic/CROPPED/001_01.bmp": "86a4c73f0fd3c121334b905a17045fbdb244f8b3df74f28775730ee0674b1cb5",
    "im_Metaplastic/CROPPED/001_02.bmp": "06c40358a14f3cb19ded9f3f844f912006cbc3bfc021230e5d6bbee8b0886e3b",
    "im_Parabasal/CROPPED/001_01.bmp": "148a22e52504907a42860cb747bb8b884a31c805eae4cb353167bd51795d443b",
    "im_Parabasal/CROPPED/001_02.bmp": "28641a6612d0d9c5b64bb1a4eebe0778b49fa77606932ee9c9c66210a314333b",
    "im_Superficial-Intermediate/CROPPED/001_01.bmp": "7775898c72c8b7c54dbc7bc29683ca6e38cc72229bd01d92576120870c76ac6a",
    "im_Superficial-Intermediate/CROPPED/001_02.bmp": "7567db4fc138be3204ab055066f4aa15d11f4020c5c567fb10795ca3aec73f07",
}

import hashlib

rusak = []
ukuran_kb = []
for f in semua_crop:
    ukuran_kb.append(f.stat().st_size / 1024)
    try:
        with Image.open(f) as im:
            im.verify()
    except Exception as e:
        rusak.append((f.name, type(e).__name__))

ukuran_kb = np.array(ukuran_kb)
total_mb = ukuran_kb.sum() / 1024

print(f"Jumlah file        : {len(ukuran_kb)}")
print(f"Total ukuran       : {total_mb:.1f} MB")
print(f"Ukuran per file    : min {ukuran_kb.min():.0f} KB | rata-rata {ukuran_kb.mean():.0f} KB | max {ukuran_kb.max():.0f} KB")
print(f"File tidak terbaca : {len(rusak)}")
for nama, jenis in rusak[:5]:
    print(f"  {nama} -> {jenis}")

assert len(rusak) == 0, f"Ada {len(rusak)} file rusak, periksa daftar di atas."

print()
print("Cek hash terhadap rilis resmi:")
cocok = beda = hilang = 0
for rel, harap in HASH_RUJUKAN.items():
    p = ROOT / rel
    if not p.exists():
        hilang += 1
        print(f"  [HILANG] {rel}")
        continue
    dapat = hashlib.sha256(p.read_bytes()).hexdigest()
    if dapat == harap:
        cocok += 1
        print(f"  [COCOK] {rel}")
    else:
        beda += 1
        print(f"  [BEDA ] {rel}")

print()
print(f"Ringkas: {cocok} cocok, {beda} beda, {hilang} tidak ada (dari {len(HASH_RUJUKAN)} file rujukan)")
if beda or hilang:
    print("PERINGATAN: ada file yang tidak sesuai rilis resmi. Periksa ulang unduhan.")
else:
    print("Semua file rujukan identik dengan rilis resmi.")

## 12. Contoh Citra per Kelas

Tiga contoh acak per kelas. Indeks sumbu mengikuti kelas yang benar-benar punya citra, sehingga tidak ada panel kosong yang menggeser judul.

In [ ]:
n_kolom = 3
kelas_ada = [r["kelas"] for r in INV if r["sel_tunggal"] > 0]

fig, axes = plt.subplots(len(kelas_ada), n_kolom, figsize=(3.4 * n_kolom, 3.4 * len(kelas_ada)))
axes = np.atleast_2d(axes)

rng = np.random.default_rng(42)
for i, nama in enumerate(kelas_ada):
    crop = sorted((ROOT / nama / "CROPPED").glob("*.bmp"))
    pilih = rng.choice(len(crop), size=min(n_kolom, len(crop)), replace=False)
    for j in range(n_kolom):
        ax = axes[i, j]
        ax.axis("off")
        if j < len(pilih):
            f = crop[pilih[j]]
            with Image.open(f) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(f.stem, fontsize=8)
    axes[i, 0].text(-0.06, 0.5, nama.replace("im_", "").replace("-", " "),
                    transform=axes[i, 0].transAxes, rotation=90,
                    va="center", ha="right", fontsize=9, fontweight="bold")

plt.suptitle("Contoh sel tunggal per kelas (SIPaKMeD)", fontweight="bold")
plt.tight_layout()
plt.savefig("sample_citra_sipakmed.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Ringkasan untuk Tabel 8

In [ ]:
ringkasan = {
    "nama_dataset": "SIPaKMeD (Single Cell Images of Cervical Cancer)",
    "sumber": "University of Ioannina, Yunani",
    "url": "https://www.cs.uoi.gr/~marina/sipakmed.html",
    "paper": "Plissiti et al. (2018), IEEE ICIP, doi:10.1109/ICIP.2018.8451389",
    "lisensi": "Untuk keperluan penelitian, wajib menyitasi Plissiti et al. (2018). Lisensi terbuka (mis. CC BY) tidak dinyatakan.",
    "cara_unduh": f"Mirror HuggingFace {REPO_HF}, hanya CROPPED/*.bmp; hash 10 file diverifikasi identik dengan rilis resmi",
    "jumlah_sel_tunggal": int(tot_crop),
    "jumlah_klaster": int(tot_klaster),
    "jumlah_kelas": len(INV),
    "per_kelas": {r["kelas"]: r["sel_tunggal"] for r in INV},
    "format": dict(format_counter),
    "resolusi_sel_tunggal": {
        "min": [int(W.min()), int(H.min())],
        "median": [int(np.median(W)), int(np.median(H))],
        "max": [int(W.max()), int(H.max())],
    },
    "ukuran_total_mb": round(float(total_mb), 1),
    "status_seimbang": status,
    "rasio_max_min": round(float(rasio), 3),
    "id_klaster_tersedia": True,
    "id_pasien_tersedia": False,
    "file_tidak_terbaca": len(rusak),
    "hash_cocok": cocok,
    "hash_beda": beda,
}

print("=" * 70)
print("HASIL VERIFIKASI UNTUK TABEL 8")
print("=" * 70)
print(f"""
1. Nama dan tautan
   {ringkasan['nama_dataset']}
   {ringkasan['sumber']}
   {ringkasan['url']}
   Paper: {ringkasan['paper']}

2. Lisensi
   {ringkasan['lisensi']}

3. Kelas dan jumlah citra
   {ringkasan['jumlah_sel_tunggal']} citra sel tunggal pada {ringkasan['jumlah_kelas']} kelas""")
for nama, n in ringkasan["per_kelas"].items():
    print(f"     - {nama.replace('im_', ''):28s} {n:4d}")
print(f"""
   Catatan: {ringkasan['jumlah_klaster']} citra klaster di luar CROPPED/ tidak dihitung
   karena populasi berbeda, bukan bagian dari 4049 citra resmi.""" if ringkasan['jumlah_klaster'] else "")
print(f"""

4. Ukuran dan format
   Total {ringkasan['ukuran_total_mb']} MB, format {', '.join(ringkasan['format'])}
   Sel tunggal: min {ringkasan['resolusi_sel_tunggal']['min'][0]}x{ringkasan['resolusi_sel_tunggal']['min'][1]} px,
                median {ringkasan['resolusi_sel_tunggal']['median'][0]}x{ringkasan['resolusi_sel_tunggal']['median'][1]} px,
                max {ringkasan['resolusi_sel_tunggal']['max'][0]}x{ringkasan['resolusi_sel_tunggal']['max'][1]} px

5. ID pasien
   Tidak tersedia. Yang tersedia hanya ID klaster (bagian 9), sehingga
   pembagian data dilakukan dengan GroupKFold per ID klaster. Potensi sel
   dari klaster sama muncul di train dan test ditekan, tetapi pembagian per
   pasien tidak mungkin dilakukan dan dinyatakan sebagai keterbatasan.

6. Subset
   Tidak diperlukan. Seluruh {ringkasan['jumlah_sel_tunggal']} citra hanya {ringkasan['ukuran_total_mb']} MB
   dan sudah seimbang ({ringkasan['status_seimbang']}, rasio {ringkasan['rasio_max_min']}:1).

7. Bukti unduhan
   {ringkasan['cara_unduh']}.
   Semua file terbaca ({ringkasan['file_tidak_terbaca']} rusak),
   hash cocok {ringkasan['hash_cocok']}/{len(HASH_RUJUKAN)}.
   Lampiran: distribusi_sipakmed.png, sample_citra_sipakmed.png,
   verifikasi_dataset.txt, verifikasi_dataset.csv
""")
print("=" * 70)

## 14. Export Hasil

`verifikasi_dataset.txt` untuk ditempel ke Tabel 8, `verifikasi_dataset.csv` untuk diolah lebih lanjut, dan `ringkasan_dataset.json` untuk dibaca pipeline berikutnya.

In [ ]:
with open("verifikasi_dataset.txt", "w", encoding="utf-8") as f:
    f.write("VERIFIKASI DATASET SIPAKMED\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Sumber   : {ringkasan['url']}\n")
    f.write(f"Paper    : {ringkasan['paper']}\n")
    f.write(f"Lisensi  : {ringkasan['lisensi']}\n\n")
    f.write(f"Sel tunggal (CROPPED/) : {ringkasan['jumlah_sel_tunggal']} citra\n")
    f.write(f"Citra klaster (root)   : {ringkasan['jumlah_klaster']} citra (tidak dipakai)\n")
    f.write(f"Jumlah kelas           : {ringkasan['jumlah_kelas']}\n\n")
    for nama, n in ringkasan["per_kelas"].items():
        f.write(f"  - {nama}: {n}\n")
    f.write(f"\nFormat          : {', '.join(ringkasan['format'])}\n")
    f.write(f"Ukuran total    : {ringkasan['ukuran_total_mb']} MB\n")
    f.write(f"Resolusi median : {ringkasan['resolusi_sel_tunggal']['median'][0]}x"
            f"{ringkasan['resolusi_sel_tunggal']['median'][1]} px\n")
    f.write(f"Resolusi rentang: {ringkasan['resolusi_sel_tunggal']['min'][0]}x"
            f"{ringkasan['resolusi_sel_tunggal']['min'][1]} sampai "
            f"{ringkasan['resolusi_sel_tunggal']['max'][0]}x"
            f"{ringkasan['resolusi_sel_tunggal']['max'][1]} px\n")
    f.write(f"Keseimbangan    : {ringkasan['status_seimbang']} (rasio {ringkasan['rasio_max_min']}:1)\n")
    f.write(f"ID pasien       : tidak tersedia\n")
    f.write(f"ID klaster      : tersedia, pakai GroupKFold\n")
    f.write(f"File rusak      : {ringkasan['file_tidak_terbaca']}\n")
    f.write(f"Hash vs resmi   : {ringkasan['hash_cocok']}/{len(HASH_RUJUKAN)} cocok\n")

with open("verifikasi_dataset.csv", "w", encoding="utf-8") as f:
    f.write("kelas,jumlah_sel_tunggal,jumlah_klaster,mb_sel_tunggal\n")
    for r in INV:
        f.write(f"{r['kelas']},{r['sel_tunggal']},{r['klaster']},{r['mb_sel_tunggal']:.1f}\n")

with open("ringkasan_dataset.json", "w", encoding="utf-8") as f:
    json.dump(ringkasan, f, indent=2, ensure_ascii=False)

print("File yang dihasilkan:")
for nama in ["distribusi_sipakmed.png", "sample_citra_sipakmed.png",
             "verifikasi_dataset.txt", "verifikasi_dataset.csv", "ringkasan_dataset.json"]:
    ada = os.path.exists(nama)
    ukuran = os.path.getsize(nama) / 1024 if ada else 0
    print(f"  [{'OK' if ada else 'X '}] {nama:28s} {ukuran:8.1f} KB")

print()
print("Verifikasi selesai. Dataset layak dipakai untuk proyek.")

if IN_COLAB:
    print()
    print("Menyimpan hasil ke Drive agar tidak hilang saat runtime mati:")
    print("  from google.colab import drive; drive.mount('/content/drive')")
    print("  !mkdir -p /content/drive/MyDrive/PCVK/hasil && cp *.png *.txt *.csv *.json /content/drive/MyDrive/PCVK/hasil/")